## 1. Installation et imports

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu

In [ ]:
import os, re, json, math, time, hashlib, unicodedata, platform, random, gc, textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pymupdf          # PyMuPDF (import "fitz" dans les anciennes versions)
import faiss

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)
print("python", platform.python_version(), "| pymupdf", pymupdf.__doc__.split()[1] if pymupdf.__doc__ else "?", "| faiss", faiss.__version__)

## 2. Configuration

In [ ]:
PDF_PATH = Path("cnil_guide_securite_personnelle.pdf")
TEST_SET_PATH = Path("test_set_b1.jsonl")
DEV_SET_PATH = Path("dev_retrieval_b1.jsonl")
OUTPUT_DIR = Path("outputs/step2_index")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# La check-list finale (pages 59-62) répète les mesures des fiches sous forme de tableau :
# l'indexer ajoute des doublons et du bruit, sans information nouvelle. Désactivée par défaut.
INDEX_CHECKLIST = False

TOP_K_MAX = 10            # profondeur de recherche évaluée

# Titres officiels des fiches (page « sommaire » du PDF). Servent d'en-têtes contextuels
# et de contrôle croisé avec les titres détectés dans le texte.
FICHE_TITLES = {
    0: "Présentation et avant-propos",
    1: "Piloter la sécurité des données",
    2: "Définir un cadre pour les utilisateurs",
    3: "Impliquer et former les utilisateurs",
    4: "Authentifier les utilisateurs",
    5: "Gérer les habilitations",
    6: "Sécuriser les postes de travail",
    7: "Sécuriser l'informatique mobile",
    8: "Protéger le réseau informatique",
    9: "Sécuriser les serveurs",
    10: "Sécuriser les sites web",
    11: "Encadrer les développements informatiques",
    12: "Protéger les locaux",
    13: "Sécuriser les échanges avec l'extérieur",
    14: "Gérer la sous-traitance",
    15: "Encadrer la maintenance et la fin de vie des matériels et logiciels",
    16: "Tracer les opérations",
    17: "Sauvegarder",
    18: "Prévoir la continuité et la reprise d'activité",
    19: "Gérer les incidents et les violations",
    20: "Analyse de risques",
    21: "Chiffrement, hachage, signature",
    22: "Cloud : informatique en nuage",
    23: "Applications mobiles : conception et développement",
    24: "Intelligence artificielle : conception et apprentissage",
    25: "API : interfaces de programmation applicative",
}
SECTION_LABELS = {
    "intro": "Introduction",
    "precautions": "Les précautions élémentaires",
    "interdits": "Ce qu'il ne faut pas faire",
    "plus_loin": "Pour aller plus loin",
    "notes": "Notes de bas de page",
    "checklist": "Check-list d'évaluation",
}

## 3. Utilitaires de texte

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", s.lower().replace("’", "'"))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s  ]+", " ", s).strip()

def clean_inline(s):
    """Nettoyage d'une ligne : ligatures (NFKC), caractères de contrôle parasites du PDF, apostrophes, espaces."""
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\x07", "").replace("­", "").replace("​", "")   # \x07 : glyphe de puce mal encodé
    s = s.replace("’", "'").replace("‘", "'")
    return re.sub(r"[ \t]+", " ", s).strip()

# Césure en fin de ligne : on recolle les mots, sauf si le début est un préfixe de mot composé.
KEEP_HYPHEN_PREFIXES = ("sous-", "lui-", "non-", "pare-", "anti-", "peer-", "cross-", "ex-", "mi-", "tout-", "post-", "co-")

def join_lines(prev, nxt):
    if prev.endswith("-") and not prev.endswith(" -") and re.match(r"[a-zàâçéèêëîïôûùüÿœ]", nxt):
        left = prev.split()[-1].lower()
        return prev + nxt if left.startswith(KEEP_HYPHEN_PREFIXES) else prev[:-1] + nxt
    return prev + " " + nxt

# Artefacts d'extraction propres à ce PDF (constatés par inspection du texte brut ; voir contrôle section 5.3).
TEXT_FIXES = {
    "soustraitan": "sous-traitan",   # « sous-traitance » : trait d'union perdu à la césure
    "luimême": "lui-même",
    "(104)": "(10^4)",               # exposants aplatis dans la note 13 (entropie)
    "(213)": "(2^13)",
}
TITLE_FIXES = {"ARTICIFICIELLE": "ARTIFICIELLE"}   # coquille du PDF dans le titre de la fiche 24

def apply_text_fixes(s):
    for a, b in TEXT_FIXES.items():
        s = s.replace(a, b)
    return s

## 4. Extraction du PDF

In [ ]:
FOOTNOTE_START = re.compile(r"^(\d{1,3}(?:\.\d)?)\s+(\S.*)$")

def page_rows(page):
    rows = []
    for b in page.get_text("dict")["blocks"]:
        for l in b.get("lines", []):
            body_spans = [s for s in l["spans"] if not (s["flags"] & 1)]       # flags & 1 : exposant
            if not body_spans:
                continue
            size = round(max(s["size"] for s in body_spans), 1)
            # dans les notes (police <= 6 pt) on garde les exposants : ce sont de vrais exposants (10^4 ...)
            if size <= 6.0:
                text = "".join(s["text"] for s in l["spans"])
            else:
                # appel de note supprimé ; on garde l'espace éventuel qui l'entourait (« CNIL17.1 dans » -> « CNIL dans »)
                text = "".join((s["text"] if not (s["flags"] & 1) else (" " if s["text"] != s["text"].rstrip() else ""))
                               for s in l["spans"])
            text = clean_inline(text)
            if text:
                rows.append({"y": l["bbox"][1], "x": l["bbox"][0], "size": size, "text": text})
    rows.sort(key=lambda r: (round(r["y"] / 4), r["x"]))   # ordre de lecture ; tolérance de 4 pt pour les cellules de tableau
    return rows

def classify_rows(rows):
    """Sépare les lignes d'une page : titres, numéro imprimé, tag, notes, tableau, corps."""
    out = {"heads": [], "printed": None, "tag": False, "notes": [], "table": [], "body": [], "sizes": set()}
    fn_idx = next((i for i, r in enumerate(rows) if r["size"] <= 6.0 and FOOTNOTE_START.match(r["text"])), None)
    for i, r in enumerate(rows):
        out["sizes"].add(r["size"])
        if r["size"] == 10.0 and re.fullmatch(r"\d{1,3}", r["text"]):
            out["printed"] = int(r["text"])
        elif r["size"] >= 13.0:
            out["heads"].append(r)
        elif r["size"] == 6.6 and r["text"].upper() == "POUR ALLER PLUS LOIN":
            out["tag"] = True
            out["tag_y"] = r["y"]
        elif fn_idx is not None and i >= fn_idx and r["size"] <= 6.0:
            out["notes"].append(r)
        elif r["size"] == 7.0:
            out["table"].append(r)
        else:
            out["body"].append(r)
    return out

doc = pymupdf.open(PDF_PATH)
PDF_SHA256 = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
raw_pages = []
for i, page in enumerate(doc):
    c = classify_rows(page_rows(page))
    c["pdf_page"] = i + 1
    raw_pages.append(c)
print(len(raw_pages), "pages | SHA-256 du PDF :", PDF_SHA256[:16], "...")

### 4.1 Numérotation des pages (citations)

In [ ]:
pairs = [(c["pdf_page"], c["printed"]) for c in raw_pages if c["printed"] is not None]
offset = Counter(p - q for p, q in pairs).most_common(1)[0][0]
anomalies = [(p, q) for p, q in pairs if p - q != offset]
print(f"Décalage dominant : page imprimée = page PDF − {offset}")
print("Anomalies de numérotation (page PDF, numéro imprimé) :", anomalies)
for c in raw_pages:
    c["page"] = c["pdf_page"] - offset          # numéro imprimé « théorique », utilisé partout ensuite

### 4.2 Type de chaque page et suivi de la fiche courante

In [ ]:
def heading_text(c):
    t = " ".join(r["text"] for r in c["heads"] if r["size"] < 30)
    for a, b in TITLE_FIXES.items():
        t = t.replace(a, b)
    return t

def page_kind(c, idx, n_pages, checklist_started):
    if idx == 0:
        return "cover"
    if idx == n_pages - 1 and not c["body"]:
        return "back_cover"
    if 7.8 in c["sizes"]:
        return "toc"
    if any(r["size"] >= 30 for r in c["heads"]) and not c["body"]:
        return "divider"
    if "ÉVALUER LE NIVEAU DE SÉCURITÉ" in heading_text(c).upper() or checklist_started:
        return "checklist"
    return "content"

checklist_started = False
for i, c in enumerate(raw_pages):
    c["kind"] = page_kind(c, i, len(raw_pages), checklist_started)
    checklist_started = checklist_started or c["kind"] == "checklist"
    # le texte d'une dernière page non classée (ex. coordonnées) ne doit pas polluer l'index
    if c["kind"] == "back_cover" or (i == len(raw_pages) - 1):
        c["kind"] = "back_cover"

# Page 2 du PDF : présentation du guide (sans titre ni numéro) ; on la rattache à la « fiche 0 »
print(pd.Series([c["kind"] for c in raw_pages]).value_counts().to_string())
fiche_heads = [(c["pdf_page"], re.match(r"FICHE\s*(\d+)", heading_text(c).upper())) for c in raw_pages if c["kind"] == "content"]
fiche_heads = [(p, int(m.group(1))) for p, m in fiche_heads if m]
assert [f for _, f in fiche_heads] == list(range(1, 26)), f"fiches détectées : {[f for _, f in fiche_heads]}"
print("25 fiches détectées, dans l'ordre. Pages PDF de début :", [p for p, _ in fiche_heads])

# contrôle croisé titres détectés / titres officiels
def fiche_title_text(c):
    """Titre de fiche seul (lignes de taille 14, éventuellement sur deux lignes)."""
    t = " ".join(r["text"] for r in c["heads"] if r["size"] == 14.0)
    for a, b in TITLE_FIXES.items():
        t = t.replace(a, b)
    return t

def _title_from_heading(h):
    return norm(re.sub(r"^FICHE\s*\d+\s*[-–—]\s*", "", h, flags=re.I))
bad_titles = []
for c in raw_pages:
    m = re.match(r"FICHE\s*(\d+)", fiche_title_text(c).upper()) if c["kind"] == "content" else None
    if m and _title_from_heading(fiche_title_text(c)).replace(" ", "") != norm(FICHE_TITLES[int(m.group(1))]).replace(" ", ""):
        bad_titles.append((int(m.group(1)), fiche_title_text(c)))
print("Titres divergents (attendu : aucun) :", bad_titles)

## 5. Segmentation en « éléments » (puces) avec métadonnées

In [ ]:
BULLET = re.compile(r"^[•·]\s*(.*)$")
NUMBERED = re.compile(r"^(\d{1,2}\.)\s+(\S.*)$")
SUBBULLET = re.compile(r"^-\s+(.*)$")
SECTION_BY_TITLE = {"les precautions elementaires": "precautions", "ce qu'il ne faut pas faire": "interdits"}

units, notes_units = [], []     # éléments du corps / éléments de notes
state = {"fiche": 0, "section": "intro"}
cur = None                      # élément en cours de construction
last_y = None

def close_unit():
    global cur
    if cur is not None and cur["text"].strip():
        cur["text"] = apply_text_fixes(re.sub(r"[ ]+", " ", cur["text"]).strip())
        units.append(cur)
    cur = None

def new_unit(text, page, y, kind):
    global cur
    close_unit()
    cur = {"fiche": state["fiche"], "section": state["section"], "kind": kind, "text": text, "pages": [page], "y": y}

page_texts = {}      # texte propre de chaque page (corps + notes) : sert au contrôle du jeu de test

for c in raw_pages:
    if c["kind"] in ("cover", "back_cover", "toc", "divider"):
        continue
    page = c["page"]
    if c["kind"] == "checklist":
        if INDEX_CHECKLIST:
            allrows = sorted(c["body"] + c["table"], key=lambda r: (round(r["y"] / 4), r["x"]))
            state_cl = {"fiche": -1, "section": "checklist"}
            close_unit()
            cur = {"fiche": -1, "section": "checklist", "kind": "table",
                   "text": " | ".join(r["text"] for r in allrows), "pages": [page], "y": 0}
            close_unit()
        continue

    # --- titres de la page : mise à jour de l'état (fiche, section)
    head_events = []
    for r in c["heads"]:
        if r["size"] >= 30:
            continue
        t = r["text"].upper()
        m = re.match(r"FICHE\s*(\d+)", t)
        if m:
            head_events.append((r["y"], "fiche", int(m.group(1))))
        elif t.startswith("AVANT-PROPOS"):
            head_events.append((r["y"], "fiche", 0))
            head_events.append((r["y"] + 0.1, "section", "intro"))
        elif norm(r["text"]) in SECTION_BY_TITLE:
            head_events.append((r["y"], "section", SECTION_BY_TITLE[norm(r["text"])]))
    if c["tag"]:
        head_events.append((c["tag_y"], "section", "plus_loin"))
    head_events.sort()

    # --- corps + tableau, dans l'ordre vertical, avec les événements de titre intercalés
    body = sorted(c["body"] + c["table"], key=lambda r: (round(r["y"] / 4), r["x"]))
    stream = [(r["y"], "row", r) for r in body] + [(y, "head", (k, v)) for y, k, v in head_events]
    stream.sort(key=lambda e: (round(e[0] / 4), 0 if e[1] == "head" else 1))
    first_row_on_page = True
    last_y = None
    in_table = False
    for y, typ, payload in stream:
        if typ == "head":
            kind_h, val = payload
            close_unit()
            if kind_h == "fiche":
                state["fiche"], state["section"] = val, "intro"
            else:
                state["section"] = val
            continue
        r = payload
        text = r["text"]
        if r["size"] == 7.0:                       # cellule de tableau (fiche 20)
            if not in_table:
                new_unit("Tableau de formalisation de l'analyse de risques : " + text, page, y, "table"); in_table = True
            else:
                cur["text"] += " | " + text
            continue
        in_table = False
        m_b, m_n, m_s = BULLET.match(text), NUMBERED.match(text), SUBBULLET.match(text)
        if text in ("]", ")") and cur is not None:    # ligne réduite à un crochet fermant (clause encadrée)
            cur["text"] += text
        elif m_b:
            new_unit(m_b.group(1), page, y, "bullet")
        elif m_n and (cur is None or cur["kind"] != "bullet" or state["fiche"] == 2):
            new_unit(m_n.group(1) + " " + m_n.group(2), page, y, "bullet")
        elif m_s:
            if cur is None:
                new_unit(m_s.group(1), page, y, "bullet")
            else:
                cur["text"] += "\n- " + m_s.group(1)
        else:
            if cur is None:
                new_unit(text, page, y, "para")
            elif first_row_on_page:
                ended = re.search(r"[.!?»]$", cur["text"]) is not None
                if ended and text[:1].isupper():
                    new_unit(text, page, y, "para")
                else:
                    cur["text"] = join_lines(cur["text"], text)
                    if page not in cur["pages"]: cur["pages"].append(page)
            elif last_y is not None and (y - last_y) > 14:
                new_unit(text, page, y, "para")
            else:
                cur["text"] = join_lines(cur["text"], text)
        first_row_on_page = False
        last_y = y
        if cur is not None and page not in cur["pages"]:
            cur["pages"].append(page)
    # un élément ne doit pas « déborder » sur une page d'une autre fiche : la fermeture se fait aux titres

    # --- notes de bas de page de cette page
    notes = []
    for r in c["notes"]:
        m = FOOTNOTE_START.match(r["text"])
        if m:
            notes.append([m.group(1), m.group(2)])
        elif notes:
            notes[-1][1] = join_lines(notes[-1][1], r["text"])
    for num, txt in notes:
        notes_units.append({"fiche": state["fiche"], "section": "notes", "kind": "note",
                            "text": apply_text_fixes(f"[{num}] {txt}"), "pages": [page], "y": 1e6})
    page_texts[page] = apply_text_fixes(
        " ".join(r["text"] for r in body) + " " + " ".join(f"[{n}] {t}" for n, t in notes))
close_unit()

print(f"{len(units)} éléments de corps | {len(notes_units)} notes | pages indexables : {len(page_texts)}")
unit_df = pd.DataFrame(units)
display(unit_df.groupby(["fiche", "section"]).agg(n=("text", "size"), chars=("text", lambda s: int(s.str.len().sum()))).unstack(fill_value=0))

### 5.1 Contrôles de cohérence de l'extraction

In [ ]:
fiches_found = sorted({u["fiche"] for u in units if u["fiche"] > 0})
assert fiches_found == list(range(1, 26)), fiches_found
for f in range(1, 26):
    secs = {u["section"] for u in units if u["fiche"] == f}
    assert "precautions" in secs, f"fiche {f} sans « précautions élémentaires »"
print("OK : 25 fiches, toutes avec section « précautions élémentaires ».")
print("Fiches sans « Ce qu'il ne faut pas faire » :", [f for f in range(1, 26) if not any(u['fiche'] == f and u['section'] == 'interdits' for u in units)])
print("Éléments très courts (< 25 car.) :", [(u['fiche'], u['pages'], u['text']) for u in units if len(u['text']) < 25][:8])
print("Éléments très longs (> 1200 car.) :", [(u['fiche'], u['pages'], len(u['text'])) for u in units if len(u['text']) > 1200])

### 5.2 Échantillon à relire à l'œil

In [ ]:
def show_unit(u, width=330):
    print(f"[fiche {u['fiche']:>2} | {u['section']:<11} | p.{','.join(map(str, u['pages']))} | {u['kind']}] "
          + textwrap.shorten(u["text"].replace("\n", " ⏎ "), width=width))

for f, sec in [(4, "precautions"), (4, "plus_loin"), (11, "precautions"), (24, "interdits")]:
    for u in [u for u in units if u["fiche"] == f and u["section"] == sec][:3]:
        show_unit(u)
print("\n--- notes (échantillon) ---")
for u in [u for u in notes_units if u["fiche"] in (4, 24, 17)][:4]:
    show_unit(u)

### 5.3 Recherche d'artefacts d'extraction restants

In [ ]:
all_text = "\n".join(u["text"] for u in units + notes_units)
tokens = Counter(re.findall(r"[a-zàâçéèêëîïôûùüÿœ]{3,}", all_text.lower()))
suspects = sorted(
    [t for t, n in tokens.items() if n == 1 and (t.endswith("nelles") or t.endswith("ment") and len(t) > 16 or t.startswith("sous") and "-" not in t)],
)
print("Mots suspects (à vérifier) :", suspects[:40])
print("Césures restantes (« mot- mot ») :", re.findall(r"\w+- \w+", all_text)[:10])